# HIT-ASR — main results on AfriSpeech-200

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/hit-asr/blob/main/notebooks/main_afrispeech.ipynb)

This notebook reproduces the main experiment of the paper on AfriSpeech-200. Every model's hyperparameters are selected on a
25 % hold-out of the corpus; every model is then trained and tested on the same ten folds of a 5×2 cross-validation
of the remaining clips, and HIT-ASR is compared with every other system on those folds.

The configuration is fixed in `hitasr.configs.MAIN["afrispeech"]` and the code is in `hitasr.experiment`. The data and the
stored searches and folds are read from the Hugging Face dataset [`huseyin-karaca/hit-asr`](https://huggingface.co/datasets/huseyin-karaca/hit-asr); nothing is written to it.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the searches and folds stored on the Hub | a CPU, minutes |
| **2** | every fold retrained from the published labels and frames (`RETUNE = True`: the search as well) | a GPU, hours |
| **3** | level 2 on labels and frames rebuilt from the audio with `extract` (set `HITASR_HUB` to that repository before the imports) | a GPU |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("hitasr") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "hit-asr[paper] @ git+https://github.com/huseyin-karaca/hit-asr.git@main"], check=True)

In [2]:
from hitasr.configs import MAIN
from hitasr.experiment import MainExperiment

LEVEL = 1          # 1, 2 or 3 (see above)
RETUNE = False     # levels 2 and 3: run the hyperparameter search again
DEVICE = "cuda"    # levels 2 and 3 train on it

exp = MainExperiment(MAIN["afrispeech"], level=LEVEL, retune=RETUNE, device=DEVICE)

AfriSpeech-200: level 1 — search read from the Hub, folds read from the Hub
experts  Cohere-Transcribe, Qwen3-ASR-1.7B, Voxtral-Mini-3B
arms     HIT-ASR, MLP-pool, ADASTT, ROVER (confidence-weighted), CN-MBR, MLP-pool (hard CE)


## 1. Hyperparameter spaces (Table 2)

The free hyperparameters of every model and their choices; every other parameter is fixed
(`hitasr.configs.SPACES`).

In [3]:
exp.spaces()

Table 2 — hyperparameter spaces
10 random draws per arm; parameters not listed are fixed (hitasr.configs.SPACES)
  [HIT-ASR  (16 free of 20)]
      parameter                                     choices
        d_model                                  {128, 256}
        n_heads                                      {4, 8}
  stage1_layers                                   {1, 2, 3}
  stage2_layers                                      {1, 2}
        ffn_dim                            {256, 512, 1024}
        dropout                                {0.05, 0.15}
         fusion           {bridge, self_attn, concat, mean}
   share_stage1                               {True, False}
     input_norm                  {none, scale, standardize}
             lr                                    {0.0001}
   weight_decay                          {0.001, 0.01, 0.1}
     batch_size                             {8, 16, 32, 64}
     max_frames                           {500, 1000, 2000}
       patience                                     {5, 10}
    loss_preset {hard_ce_only, soft_ce_only, wer+hard, all}
label_smoothing                                  {0.0, 0.1}
  [MLP-pool  (8 free of 12)]
      parameter                        choices
       d_hidden         {256, 512, 1024, 2048}
       n_layers                      {1, 2, 3}
        dropout              {0.05, 0.15, 0.3}
             lr {3e-05, 0.0001, 0.0003, 0.001}
   weight_decay             {0.001, 0.01, 0.1}
     batch_size             {32, 64, 128, 256}
       patience                        {5, 10}
label_smoothing                     {0.0, 0.1}
  [ADASTT  (7 free of 8)]
       parameter                choices
    n_estimators       {100, 400, 1000}
   learning_rate {0.01, 0.03, 0.1, 0.3}
       max_depth              {3, 5, 8}
       subsample        {0.6, 0.8, 1.0}
colsample_bytree   {0.1, 0.3, 0.6, 1.0}
min_child_weight       {1.0, 4.0, 16.0}
      reg_lambda       {0.3, 3.0, 30.0}
  [ROVER (confidence-weighted)  (6 free of 6)]
     parameter                          choices
         alpha                            {0.0}
   conf_source               {cn, probe, blend}
     weighting {uniform, inverse_wer, log_odds}
weight_by_conf                    {False, True}
     conf_temp                            {2.0}
  null_penalty                      {0.25, 0.5}
  [CN-MBR  (7 free of 7)]
  parameter                          choices
     decode                      {slot, hyp}
 candidates              {members+consensus}
      prior {uniform, inverse_wer, log_odds}
temperature                           {10.0}
 null_prior                      {0.05, 0.2}
   use_conf                    {False, True}
conf_source                          {probe}
  [MLP-pool (hard CE)  (8 free of 12)]
      parameter                        choices
       d_hidden         {256, 512, 1024, 2048}
       n_layers                      {1, 2, 3}
        dropout              {0.05, 0.15, 0.3}
             lr {3e-05, 0.0001, 0.0003, 0.001}
   weight_decay             {0.001, 0.01, 0.1}
     batch_size             {32, 64, 128, 256}
       patience                        {5, 10}
label_smoothing                     {0.0, 0.1}

## 2. Data

The three experts' transcripts and word-error counters for every clip (at levels 2 and 3, also their frame-level
encoder states). The stored counters are recomputed from the stored transcripts as a check.

In [4]:
exp.load()

afrispeech: 9,544 utterances x 3 experts (0 adopted from fastt) in 2.4s
100.0% of 1,200 (utterance, model) counters reproduce exactly


RouterStore(afrispeech, cohere_transcribe+qwen3_asr_1_7b+voxtral_mini_3b, 9,544 rows, frames closed)

## 3. Hyperparameter search

A random 25 % of the clips is held out for the search. For every model, ten configurations are drawn at random from
its space (Table 2), each trained on 70 % of the hold-out and scored on the remaining 30 %; the configuration with the
lowest corpus WER is kept. The other 75 % of the clips are never seen by the search.

In [5]:
exp.search()

partition,n_utts,share
tune,2386,0.250
eval,7158,0.750
fit,1670,0.175
val,716,0.075


  on the tuning-score rows: bsm(qwen3_asr_1_7b) 0.2369   oracle 0.1626   gap 0.0743
  the 7,158 evaluation rows are never seen by a trial
  HIT-ASR 10 trials; MLP-pool 10 trials; ADASTT 10 trials; ROVER (confidence-weighted) 10 trials; CN-MBR 10 trials; MLP-pool (hard CE) 10 trials — read from the Hub


HPTRun('hpt_afrispeech_e1b1792c', 6 arm(s), best mlp_pool 0.2072, eval on 7,158 rows)

In [6]:
exp.dataset()

statistic,afrispeech
Total clips,9544
Hyperparameter hold-out,2386
Evaluation rows (5x2),7158
Train size (per fold),3579
Test size (per fold),3579
Hours,27.3864
Mean clip length (s),10.3302
Reference words,156221


In [7]:
exp.selected()

label,trials,hold-out WER,selected configuration
HIT-ASR,10,0.2098,"{""batch_size"": 8, ""d_model"": 256, ""dropout"": 0.05, ""ffn_dim"": 1024, ""fusion"": ""concat"", ""input_norm"": ""none"", ""label_smoothing"": 0.1, ""loss_preset"": ""soft_ce_only"", ""lr"": 0.0001, ""max_frames"": 2000, ""max_restarts"": 0, ""n_heads"": 8, ""n_seeds"": 3, ""patience"": 5, ""pooled_skip"": false, ""share_stage1"": false, ""stage1_layers"": 3, ""stage2_layers"": 2, ""weight_decay"": 0.1, ""word_weighted"": false}"
MLP-pool,10,0.2072,"{""batch_size"": 64, ""d_hidden"": 512, ""dropout"": 0.3, ""label_smoothing"": 0.1, ""loss_preset"": ""all"", ""lr"": 0.0001, ""max_restarts"": 0, ""n_layers"": 1, ""n_seeds"": 3, ""patience"": 5, ""weight_decay"": 0.01, ""word_weighted"": false}"
ADASTT,10,0.2140,"{""colsample_bytree"": 0.3, ""learning_rate"": 0.3, ""max_depth"": 8, ""min_child_weight"": 16.0, ""n_estimators"": 400, ""objective"": ""cross_entropy"", ""reg_lambda"": 0.3, ""subsample"": 0.8}"
ROVER (confidence-weighted),10,0.2611,"{""alpha"": 0.0, ""conf_source"": ""probe"", ""conf_temp"": 2.0, ""null_penalty"": 0.5, ""weight_by_conf"": false, ""weighting"": ""inverse_wer""}"
CN-MBR,10,0.2702,"{""candidates"": ""members+consensus"", ""conf_source"": ""probe"", ""decode"": ""slot"", ""null_prior"": 0.2, ""prior"": ""inverse_wer"", ""temperature"": 10.0, ""use_conf"": false}"
MLP-pool (hard CE),10,0.2114,"{""batch_size"": 64, ""d_hidden"": 256, ""dropout"": 0.3, ""label_smoothing"": 0.1, ""loss_preset"": ""hard_ce_only"", ""lr"": 0.001, ""max_restarts"": 0, ""n_layers"": 3, ""n_seeds"": 3, ""patience"": 5, ""weight_decay"": 0.1, ""word_weighted"": false}"


## 4. 5×2 cross-validation

Five repetitions, each splitting the evaluation clips into two halves with its own seed. Every model is trained on
one half and tested on the other, then the other way round: ten folds, the same for every model.

In [8]:
exp.cross_validate();

  hub: runlog main_afrispeech — 35 of 35 (seed, group) result(s) available; fetched 70 file(s) in 1s
  runlog HIT  cv5x2 seed 2024 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 123 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 777 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 3 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 86400 — 7/7 group(s) cached, nothing refitted
5 repetitions x 2 folds x 15 arms


## 5. Main results (Table 3)

In [9]:
exp.results()

Table 3 — main results, AfriSpeech-200
mean ± sd over 10 folds; WER and uWER (mean utterance WER) as fractions, GC (share of the gap between the best single expert and the oracle closed) and Acc (selection accuracy) in %; dist = share of clips routed to Cohere-Transcribe / Qwen3-ASR-1.7B / Voxtral-Mini-3B
  [expert]
                                arm             WER            uWER        GC@0      Acc@0           dist
                  Cohere-Transcribe 0.2590 ± 0.0119 0.3056 ± 0.0110  -1.0 ± 7.8 60.6 ± 0.5 1.00-0.00-0.00
                     Qwen3-ASR-1.7B 0.2523 ± 0.0032 0.2969 ± 0.0056   6.1 ± 9.8 51.6 ± 0.4 0.00-1.00-0.00
                    Voxtral-Mini-3B 0.2625 ± 0.0075 0.3191 ± 0.0105 -6.2 ± 11.4 46.9 ± 0.9 0.00-0.00-1.00
Best single expert (Qwen3-ASR-1.7B) 0.2582 ± 0.0103 0.3006 ± 0.0104   0.0 ± 0.0 54.3 ± 4.6 0.30-0.70-0.00
  [reference]
   arm             WER            uWER      GC@0      Acc@0           dist
Random 0.2569 ± 0.0094 0.3081 ± 0.0169 1.1 ± 8.4 53.2 ± 0.4 0.33-0.33-0.33
  [fusion]
                        arm             WER            uWER         GC@0 Acc@0 dist
ROVER (confidence-weighted) 0.3117 ± 0.0127 0.3579 ± 0.0088 -65.1 ± 13.3     —    —
                     CN-MBR 0.3171 ± 0.0136 0.3783 ± 0.0154 -71.6 ± 11.9     —    —
  [pooled]
     arm             WER            uWER        GC@0      Acc@0           dist
MLP-pool 0.2371 ± 0.0079 0.2767 ± 0.0090  24.8 ± 8.9 62.9 ± 0.6 0.63-0.24-0.13
  ADASTT 0.2484 ± 0.0136 0.2916 ± 0.0147 11.8 ± 11.5 61.6 ± 0.5 0.87-0.10-0.03
  [ours]
    arm             WER            uWER       GC@0      Acc@0           dist
HIT-ASR 0.2293 ± 0.0059 0.2729 ± 0.0075 34.1 ± 7.6 63.4 ± 0.9 0.53-0.28-0.20
  [oracle]
                      arm             WER            uWER        GC@0       Acc@0           dist
                   Oracle 0.1751 ± 0.0038 0.2059 ± 0.0053 100.0 ± 0.0 100.0 ± 0.0 0.61-0.25-0.14
Oracle (1-word tolerance) 0.1854 ± 0.0036 0.2211 ± 0.0053  87.5 ± 1.3  83.1 ± 0.4 0.39-0.52-0.09
Oracle (2-word tolerance) 0.2021 ± 0.0038 0.2443 ± 0.0057  67.0 ± 6.0  69.3 ± 2.1 0.34-0.61-0.05

In [10]:
exp.routing()

router,Cohere-Transcribe,Qwen3-ASR-1.7B,Voxtral-Mini-3B,entropy,switch rate
Random,33.4,33.4,33.2,1.585,66.5
MLP-pool,63.1,24.4,12.6,1.292,63.7
ADASTT,87.0,10.0,3.0,0.657,67.2
HIT-ASR,52.8,27.6,19.6,1.460,63.6
Oracle,60.6,24.9,14.4,1.340,64.3
Oracle (1-word tolerance),38.9,52.1,8.9,1.331,28.7
Oracle (2-word tolerance),34.4,60.5,5.0,1.185,14.9


## 6. Statistical test (Table 6)

HIT-ASR against every other system on the same ten folds, with a custom statistical test. For full details, please
see the paper.

In [11]:
exp.tests()

Table 6 — HIT-ASR against every system, AfriSpeech-200
on the same 10 folds; effect in the metric's own units
anchor: HIT-ASR; positive effect = anchor better
  [corpus WER]
                 comparator  effect       t      p
          Cohere-Transcribe +0.0296   2.558 0.0508
             Qwen3-ASR-1.7B +0.0230   6.773 0.0011
            Voxtral-Mini-3B +0.0332   8.937 0.0003
         Best single expert +0.0289   2.628 0.0467
                     Random +0.0276   3.838 0.0121
ROVER (confidence-weighted) +0.0823   5.643 0.0024
                     CN-MBR +0.0878   6.119 0.0017
                   MLP-pool +0.0078   2.653 0.0453
                     ADASTT +0.0191   1.681 0.1536
                     Oracle -0.0542 -12.283 0.0001
  Oracle (1-word tolerance) -0.0440  -9.705 0.0002
  Oracle (2-word tolerance) -0.0273  -3.786 0.0128
  [mean utterance WER]
                 comparator  effect       t      p
          Cohere-Transcribe +0.0326   4.069 0.0096
             Qwen3-ASR-1.7B +0.0240   9.837 0.0002
            Voxtral-Mini-3B +0.0462   8.768 0.0003
         Best single expert +0.0276   3.422 0.0188
                     Random +0.0352   2.826 0.0369
ROVER (confidence-weighted) +0.0850   9.245 0.0002
                     CN-MBR +0.1054   7.753 0.0006
                   MLP-pool +0.0038   2.046 0.0961
                     ADASTT +0.0187   1.153 0.3009
                     Oracle -0.0670 -13.283 0.0000
  Oracle (1-word tolerance) -0.0518  -9.723 0.0002
  Oracle (2-word tolerance) -0.0286  -4.006 0.0103
Custom statistical test — for full details, please see the paper.

## 7. Record

The configuration, the selected hyperparameters and every table above, in `main_afrispeech.json`.

In [12]:
exp.save();

wrote main_afrispeech.json
